# 00 Data Aggregation Inventory

This notebook is the starting point for Stage 1A: data aggregation. It documents the actual datasets we are mining, inventories what currently exists in `data/raw`, and gives runnable cells for refreshing raw source snapshots.

The important rule: raw data is evidence. We snapshot it first, then parse, harmonize, clean, and impute in later stages.

In [ ]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd if (cwd / 'data').exists() else cwd.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
SOURCE_MANIFEST = PROJECT_ROOT / 'data' / 'source_manifest.csv'

PROJECT_ROOT

## Actual Raw Dataset Families

| Dataset family | Primary raw paths | Role |
| --- | --- | --- |
| Fjelstul World Cup Database | `data/raw/fjelstul_*.csv` | Historical World Cup backbone: tournaments, teams, matches, squads, events, standings |
| International results | `data/raw/international_matches.csv`, `match_goalscorers.csv`, `match_shootouts.csv` | All international match history for rating rebuilds and non-World-Cup context |
| Transfermarkt player data | `data/raw/transfermarkt_*.csv` | Player, club, appearance, and market-value context |
| Wikipedia squad pages | `data/raw/world_cup_squads.csv` when scraped | Squad context enrichment: DOB, caps, goals, clubs, positions |
| 2026 live overlay | `data/raw/2026_live/YYYY-MM-DD/run-*/...` | Current 2026 fixtures, venues, squads/context, official/source snapshots |
| Curated tournament metadata | `data/reference/tournament_metadata.csv` | Hand-checked tournament-level fields including 2026 placeholders |

In [ ]:
manifest = pd.read_csv(SOURCE_MANIFEST)
manifest

## Inventory `data/raw`

This inventories the raw files without loading every large dataset into memory. CSV row counts are approximate line counts minus the header.

In [ ]:
def count_csv_rows(path: Path) -> int | None:
    if path.suffix.lower() != '.csv':
        return None
    with path.open('rb') as handle:
        line_count = sum(1 for _ in handle)
    return max(line_count - 1, 0)

rows = []
for path in sorted(RAW_DIR.rglob('*')):
    if path.is_file():
        rows.append({
            'relative_path': str(path.relative_to(PROJECT_ROOT)),
            'suffix': path.suffix.lower(),
            'size_mb': round(path.stat().st_size / (1024 * 1024), 3),
            'rows_if_csv': count_csv_rows(path),
        })

raw_inventory = pd.DataFrame(rows)
raw_inventory

## Pull Or Refresh Raw Data

Run individual collectors when you want a controlled refresh. Use `collect_all.py` only when you intentionally want to refresh every source family.

In [ ]:
# Historical World Cup relational backbone
# !python -m src.data_collection.collect_worldcup_db

In [ ]:
# International match results, goalscorers, and shootouts
# !python -m src.data_collection.collect_matches

In [ ]:
# Wikipedia squad context scrape, including 2026 if the page is populated
# !python -m src.data_collection.collect_squads

In [ ]:
# Transfermarkt player/club/appearance/valuation data
# !python -m src.data_collection.collect_player_stats

In [ ]:
# 2026 live/provisional overlay snapshots
# !python -m src.data_collection.collect_2026_live

## Inspect Latest 2026 Live Snapshot

In [ ]:
live_root = RAW_DIR / '2026_live'
run_dirs = sorted([p for p in live_root.glob('*/*') if p.is_dir()]) if live_root.exists() else []

if run_dirs:
    latest_run = run_dirs[-1]
    print(latest_run.relative_to(PROJECT_ROOT))
    with (latest_run / 'summary.json').open() as handle:
        summary = json.load(handle)
    display(summary)
else:
    print('No 2026 live snapshots yet. Run: python -m src.data_collection.collect_2026_live')